[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/33_kv_cache_solution.ipynb)

# 🟡 Solution: KV Cache for Incremental Decoding (nnx.Module)

*Attention & Transformers · Medium*

Reference implementation. Try it yourself in `33_kv_cache.ipynb` first.

---
Implement a **preallocated key/value cache** for autoregressive decoding, as an
`nnx.Module` that owns mutable state.

### Rules
- Signature: `KVCache(batch_size, num_heads, max_len, head_dim, *, rngs=None)`
- `self.k_cache`, `self.v_cache`: `(batch_size, num_heads, max_len, head_dim)` zeros,
  wrapped in `nnx.Cache` (or any `nnx.Variable` that is **not** `nnx.Param` — the
  optimizer must never see them)
- `self.pos`: a scalar `int32` write position, starting at `0`, also cached state
- `update(k_new, v_new)` takes `(B, H, S, Dh)` — `S = 1` for a decode step, `S > 1`
  for prefill — writes them at `pos`, advances `pos` by `S`, and returns
  `(keys, values, mask)`
- `keys`/`values` are always the **full** `(B, H, max_len, Dh)` buffers; `mask` is
  a `(max_len,)` boolean marking the written positions
- Write with `jax.lax.dynamic_update_slice_in_dim`. No `jnp.concatenate`, no
  Python-level growth, no `nnx.MultiHeadAttention`

### Why the return value is a mask and not a slice
"Return the valid slice" is the obvious API and it is unimplementable under
`jit`. In JAX a shape is part of the type, so `keys[:, :, :self.pos]` with a
traced `pos` is a shape that depends on a value — an error, not a slow path. The
jit-compatible spelling of "the first `pos` entries" is a fixed-size buffer plus
a boolean mask, which the caller folds into the scores:

```python
scores = jnp.where(mask, scores, -jnp.inf)   # then softmax
```

The masked-out slots are still multiplied, so you pay for `max_len` columns from
step one. That is the deliberate trade: a *constant* amount of wasted flops in
exchange for one compilation. Concatenating instead gives `(B, H, 1, Dh)`,
`(B, H, 2, Dh)`, ... — a **new shape, therefore a new XLA program, on every
decode step**. Generating 4,096 tokens means 4,096 compilations, each of which
costs far more than the whole forward pass. Preallocation is not an optimization
here; it is the difference between working and not.

The buffers are `nnx.Cache` rather than `nnx.Param` for the same reason
BatchNorm's running stats are `nnx.BatchStat`: `nnx.state(model, nnx.Param)`
must hand the optimizer parameters only. And because a write mutates that state,
`jax.grad` over `update` raises `TraceContextError` — use `nnx.grad`, with
`argnums=` to differentiate w.r.t. the incoming tensors.

### The memory arithmetic
A cache holds two tensors per layer for every token generated so far:

$$\text{bytes} = 2 \cdot L \cdot H_{kv} \cdot d_h \cdot T \cdot \text{sizeof(dtype)}$$

Llama-3-70B in bf16: $L = 80$, $H_{kv} = 8$ (grouped-query), $d_h = 128$, so
$2 \cdot 80 \cdot 8 \cdot 128 \cdot 2 = 327{,}680$ bytes $= 320$ KiB **per
token**. At 32k context that is ~10.7 GB for a *single* sequence; a batch of 32
needs ~344 GB, well past the 140 GB the weights themselves occupy. Past a few
thousand tokens the cache, not the model, is what caps your batch size — and
batch size is throughput. This is why GQA exists
(64 query heads sharing 8 KV heads cuts that 320 KiB by 8x), why people quantize
the cache to int8, and why vLLM's PagedAttention allocates fixed-size blocks
instead of `max_len` per sequence: preallocating for the worst case wastes
everything a short sequence never uses.

The cache also changes the compute asymptotics. Re-running full attention over
the whole prefix at every step costs $O(T^2)$ per step and $O(T^3)$ per
sequence; with a cache each step is $O(T)$ and the sequence is $O(T^2)$.

### Gotcha
`dynamic_update_slice` **clamps** an out-of-range start index instead of raising.
Write past `max_len` and it silently overwrites the last slots — no error, just
wrong answers. Bound the write position yourself in production code.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


class KVCache(nnx.Module):
    def __init__(self, batch_size: int, num_heads: int, max_len: int,
                 head_dim: int, *, rngs: nnx.Rngs = None):
        shape = (batch_size, num_heads, max_len, head_dim)
        # nnx.Cache is a non-Param Variable: mutable state the optimizer ignores.
        self.k_cache = nnx.Cache(jnp.zeros(shape))
        self.v_cache = nnx.Cache(jnp.zeros(shape))
        self.pos = nnx.Cache(jnp.array(0, dtype=jnp.int32))
        # Static python ints — safe to use in shapes.
        self.max_len = max_len
        self.batch_size = batch_size
        self.num_heads = num_heads
        self.head_dim = head_dim

    def update(self, k_new, v_new):
        s = k_new.shape[2]          # static: it comes from a shape
        p = self.pos[...]          # traced: it comes from state

        # In-place write at a dynamic offset, keeping the buffer shape constant.
        self.k_cache[...] = jax.lax.dynamic_update_slice_in_dim(
            self.k_cache[...], k_new, p, axis=2
        )
        self.v_cache[...] = jax.lax.dynamic_update_slice_in_dim(
            self.v_cache[...], v_new, p, axis=2
        )
        self.pos[...] = p + s

        # "The valid slice", expressed so the shape stays static.
        mask = jnp.arange(self.max_len) < self.pos[...]
        return self.k_cache[...], self.v_cache[...], mask

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp
from flax import nnx

B, H, Dh, MAX = 1, 2, 4, 8
cache = KVCache(B, H, MAX, Dh)
key = jax.random.key(0)

print("pos:", cache.pos[...], " buffer:", cache.k_cache[...].shape)

# Prefill 3 tokens, then decode 2 more one at a time.
k, v = jax.random.normal(key, (2, B, H, 3, Dh))
keys, values, mask = cache.update(k, v)
print("after prefill  pos=", cache.pos[...], "mask=", mask)

for step in range(2):
    k1, v1 = jax.random.normal(jax.random.key(step + 1), (2, B, H, 1, Dh))
    keys, values, mask = cache.update(k1, v1)
    # Shape is identical every step — that is the whole point.
    print(f"after decode {step}  keys.shape={keys.shape}  valid={int(mask.sum())}")

# 320 KiB/token for Llama-3-70B in bf16:
per_tok = 2 * 80 * 8 * 128 * 2
print(f"\nLlama-3-70B cache: {per_tok / 1024:.0f} KiB/token "
      f"-> {per_tok * 32768 / 1e9:.1f} GB at 32k context")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("kv_cache")